In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import time
from collections import OrderedDict
from pathlib import Path
import sys

# ==========================================
# 0. PARAMÈTRES DE TON ÉQUIPE ET ENVIRONNEMENT
# ==========================================
if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    # Utilisation de Path pour unifier les slashs
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"  # Gestion automatique des chemins Linux/Windows

from mpp_project.oracle_dp import compute_full_Q_table
from mpp_project.core import build_exact_score_market, calculate_true_outcome_probas_from_odds, load_exact_scores_by_match
from mpp_project.daily_pipeline import eval_exact_market
from mpp_project.end_game_solver import GAP_MIN, GAP_MAX, GAP_OFFSET

# --- PARAMÈTRES DU JOUR ---
MATCH_DU_JOUR_ID = 73    
match_idx_global = MATCH_DU_JOUR_ID - 1
match_idx_pf = match_idx_global - 72 

MON_GAP_1 = 0
MON_GAP_2 = 0
HAS_BOOSTER = 1  # 1 = Booster dispo, 0 = Déjà utilisé
HORIZON_NUIT = 4 # Nombre de matchs à exporter pour l'appli mobile

# --- ÉTAT DES FAVORIS ET BUTEURS (V_term) ---
my_fav_alive = 1 
bob_fav_alive = 1
pack_fav_alive = 1

# --- SCORES EXACTS (data/exact_scores.csv) ---
# Mêmes données et mêmes tableaux que le NB10, appliqués aux phases finales.
# Ajoute les lignes des matchs de phase finale (match_id >= 73) dans le CSV.
# NB : l'horizon des phases finales (expected_V_phases_finales_full.npy) reste 1N2 ;
# l'effet scores-exacts porte sur le payoff + bonus du match évalué (pas propagé
# dans la valeur de continuation, qui demanderait de régénérer l'horizon offline).
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")

print(f"🎯 Match cible Global : {MATCH_DU_JOUR_ID} | Index Python Phase Finale : {match_idx_pf}")
print(f"📋 Matchs renseignés dans exact_scores.csv : {sorted(exact_scores_by_match)}")

In [15]:
# ==========================================
# 1. LECTURE DE L'HORIZON ET GÉNÉRATION DES ABAQUES (Q-TABLES)
# ==========================================
print("Chargement des données du tournoi...")
df_full = pd.read_csv(DATA_DIR / "CDM_2026.csv")

print("Chargement de la Thermodynamique du Peloton (Générée par le Notebook 15)...")
p_empirique_1D = np.load(DATA_DIR / "p_empirique_1D.npy")
max_gain_dynamique = p_empirique_1D.shape[2]

print("Chargement de l'Horizon Glissant (Généré la nuit par bracket_simulator.py)...")
V_all_matches = np.load(DATA_DIR / "expected_V_phases_finales_full.npy")

print("\n" + "="*50)
print(f"Génération des Abaques pour les prochains matchs...")
print("="*50)

for k in range(HORIZON_NUIT):
    t_pf = match_idx_pf + k
    if t_pf >= 32: break
        
    match_id_cible_global = t_pf + 73 
    t_global = match_id_cible_global - 1
    
    row = df_full.iloc[t_global]
    
    # Arrêt si on atteint des matchs non encore ouverts (ex: on ne connait pas l'affiche)
    if pd.isna(row.get('cote_1', np.nan)):
        print(f"⚠️ Cotes non renseignées pour le match {match_id_cible_global}. Fin de la génération.")
        break

    # 1. Extraction des données réelles du match
    inv = 1.0 / np.array([row['cote_1'], row['cote_N'], row['cote_2']])
    t_prob = inv / np.sum(inv)
    t_gains = np.array([row['gain_mpp_1'], row['gain_mpp_N'], row['gain_mpp_2']], dtype=np.int32)
    c_brut = np.array([row['crowd_1'], row['crowd_N'], row['crowd_2']], dtype=np.float32)
    t_crowds = c_brut / np.sum(c_brut)

    # 2. Récupération de l'état futur depuis la matrice V générée la nuit
    if t_pf + 1 < 32:
        # V_all_matches contient l'évolution des 32 matchs de phases finales
        V_next_t = V_all_matches[t_pf + 1, :, :, :, my_fav_alive, bob_fav_alive, pack_fav_alive]
    else:
        # Cas exceptionnel si on calcule le tout dernier match (La Finale)
        V_term_base = np.load(DATA_DIR / "V_term_buteurs.npy")
        V_next_t = np.zeros((1001, 1001, 2), dtype=np.float32)
        for g1 in range(1001):
            idx1 = min(500, int(round(g1 / 2.0)))
            for g2 in range(1001):
                idx2 = min(500, int(round(g2 / 2.0)))
                val = V_term_base[idx1, idx2, my_fav_alive, bob_fav_alive, pack_fav_alive]
                V_next_t[g1, g2, 0] = val
                V_next_t[g1, g2, 1] = val

    print(f"Calcul de l'Abaque pour le Match {match_id_cible_global}...")
    
    # On passe t_global pour que la Q-table pioche la bonne tranche de p_empirique_1D
    Q_table_t = compute_full_Q_table(
        t=t_global, 
        t_prob=t_prob, 
        c_rep=t_crowds, 
        t_gains=t_gains, 
        p_empirique_1D=p_empirique_1D, 
        alpha=1.0,  # En phases finales, alpha=1.0 car le peloton est dilué
        V_next=V_next_t, 
        max_gain=max_gain_dynamique 
    )

    export_path = DATA_DIR / f"Abaque_Match_{match_id_cible_global}.npz"
    np.savez_compressed(export_path, q_table=Q_table_t, ev_actions=t_prob * t_gains)

print(f"\n✅ Terminé ! Fichiers synchronisés dans {DATA_DIR}.")

Chargement des données du tournoi...
Chargement de la Thermodynamique du Peloton (Générée par le Notebook 15)...
Chargement de l'Horizon Glissant (Généré la nuit par bracket_simulator.py)...

Génération des Abaques pour les prochains matchs...
Calcul de l'Abaque pour le Match 73...
Calcul de l'Abaque pour le Match 74...
Calcul de l'Abaque pour le Match 75...
Calcul de l'Abaque pour le Match 76...

✅ Terminé ! Fichiers synchronisés dans d:\Documents\Code\MonPetitPronoStrategy\data.


In [ ]:
# ==========================================
# 2. SCORES EXACTS — TABLEAUX PAR MATCH (phases finales)
# ==========================================
# Mêmes tableaux que le NB10 : pour chaque match de phase finale de l'horizon
# présent dans exact_scores.csv, on évalue le marché des scores exacts à l'état de
# gap courant, chaîné sur l'horizon endgame (tranché par l'état des favoris).
# V_next reste 1N2 (cf. caveat cellule 0) ; les bonus jouent sur le match évalué.

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(MON_GAP_1)))) + GAP_OFFSET
g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(MON_GAP_2)))) + GAP_OFFSET

# Équipes pour l'entête (clé = position CDM_2026.csv + 1 = match_id global)
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(df_full.itertuples(index=False))
}


def _v_next_pf(t_pf):
    """V_next (1001,1001,2) pour le match de phase finale t_pf, tranché par l'état favoris."""
    if t_pf + 1 < 32:
        return V_all_matches[t_pf + 1, :, :, :, my_fav_alive, bob_fav_alive, pack_fav_alive]
    # Finale : terminal buteurs (upsamplé en demi-points -> plein)
    V_term_base = np.load(DATA_DIR / "V_term_buteurs.npy")
    Vn = np.zeros((1001, 1001, 2), dtype=np.float32)
    for g1 in range(1001):
        i1 = min(500, int(round(g1 / 2.0)))
        for g2 in range(1001):
            i2 = min(500, int(round(g2 / 2.0)))
            v = V_term_base[i1, i2, my_fav_alive, bob_fav_alive, pack_fav_alive]
            Vn[g1, g2, 0] = v
            Vn[g1, g2, 1] = v
    return Vn


# --- Construction des marchés de la nuit (phase finale) ---
night_markets = OrderedDict()
for k in range(HORIZON_NUIT):
    t_pf = match_idx_pf + k
    if t_pf >= 32:
        break
    match_id = t_pf + 73
    t_global = match_id - 1
    if match_id not in exact_scores_by_match:
        continue
    row = df_full.iloc[t_global]
    if pd.isna(row.get('cote_1', np.nan)):
        break
    odds = np.array([row['cote_1'], row['cote_N'], row['cote_2']], dtype=float)
    outcome_probas = calculate_true_outcome_probas_from_odds(odds)   # 1N2 Shin (ancrage)
    market = build_exact_score_market(exact_scores_by_match[match_id], outcome_probas=outcome_probas)
    gains = np.array([row['gain_mpp_1'], row['gain_mpp_N'], row['gain_mpp_2']], dtype=float)
    crowd = np.array([row['crowd_1'], row['crowd_N'], row['crowd_2']], dtype=float)
    crowd = crowd / crowd.sum()
    V_next = _v_next_pf(t_pf)
    reco_m, wr_m, df_m = eval_exact_market(
        market, g1_idx, g2_idx, HAS_BOOSTER,
        gains, crowd, p_empirique_1D[t_global], 1.0, V_next, noms_choix
    )
    night_markets[match_id] = (reco_m, wr_m, df_m)


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)

    fmt = {}
    for c in view.columns:
        if c == "E[pts 1/2/3]":
            fmt[c] = "{:.3f}"
        elif c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


if not night_markets:
    print("⚠️ Aucun match de phase finale de l'horizon n'est renseigné dans exact_scores.csv "
          f"(matchs {MATCH_DU_JOUR_ID}..{MATCH_DU_JOUR_ID + HORIZON_NUIT - 1}). "
          "Ajoute leurs lignes (match_id, score, cote, crowd) pour obtenir les tableaux.")
else:
    for match_id, (reco_m, wr_m, df_m) in night_markets.items():
        _afficher_marche(match_id, reco_m, wr_m, df_m)

In [17]:
# ==========================================
# 3. PROJECTION STRATÉGIQUE (MATCH DU JOUR & SUIVANTS)
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE (PHASES FINALES)")
print("="*50)
print("Analyse de sensibilité : Évolution de la stratégie selon la dynamique\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-30 pts/match)", "delta": -30},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+30 pts/match)", "delta": 30}
]

for k in range(HORIZON_NUIT):
    t_pf = match_idx_pf + k
    if t_pf >= 32: break
        
    match_id_cible_global = t_pf + 73 
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible_global}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        break
        
    print(f"▶️ MATCH {match_id_cible_global} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            
            best_keep_idx = np.argmax(wr_keep)
            best_use_idx = np.argmax(wr_use)
            
            val_keep = wr_keep[best_keep_idx]
            val_use = wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
                
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE (PHASES FINALES)
Analyse de sensibilité : Évolution de la stratégie selon la dynamique

▶️ MATCH 73 (Δt = +0) :
  🔴 Retard (-30 pts/match)    | Gaps proj. :    0,    0 | Reco : 2 (Ext) (Safe) | WR(Safe): 31.23% | WR(x2): 25.92%
  ⚪ Maintien (Inchangé)       | Gaps proj. :    0,    0 | Reco : 2 (Ext) (Safe) | WR(Safe): 31.23% | WR(x2): 25.92%
  🟢 Avance (+30 pts/match)    | Gaps proj. :    0,    0 | Reco : 2 (Ext) (Safe) | WR(Safe): 31.23% | WR(x2): 25.92%
------------------------------------------------------------------------------------------
▶️ MATCH 74 (Δt = +1) :
  🔴 Retard (-30 pts/match)    | Gaps proj. :  -30,  -30 | Reco : 1 (Dom) (Safe) | WR(Safe): 27.94% | WR(x2): 23.64%
  ⚪ Maintien (Inchangé)       | Gaps proj. :    0,    0 | Reco : 1 (Dom) (Safe) | WR(Safe): 30.60% | WR(x2): 26.30%
  🟢 Avance (+30 pts/match)    | Gaps proj. :   30,   30 | Reco : 1 (Dom) (Safe) | WR(Safe): 33.49% | WR(x2): 29.19%
---------------------------------------------------